# 00 - Step 0: Setup

**GeoAI LAB - Soil Texture from Spectra**

Run this notebook once, before section 1. It does five things:

1. Loads the three CSVs and **asserts** their row alignment instead of trusting it.
2. Defines the spectral column group `SPEC` and the wavelength axis `WL`.
3. Documents what each file is for.
4. Caches everything to Parquet, so later notebooks reload in ~0.2 s from a 38 MB file, instead of re-parsing 139 MB of CSV.
5. Creates `figs/` and a `save_fig()` helper that enforces the *one file per task* naming rule.

It also records two facts that section 5 depends on. Nothing here is a lab deliverable.

## 0.0 - Imports and paths

In [ ]:
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Works whether the kernel starts in notebooks/ or in the project root.
ROOT = Path.cwd()
if not (ROOT / "ossl_tp_dataset.csv").exists():
    ROOT = ROOT.parent
assert (ROOT / "ossl_tp_dataset.csv").exists(), f"data not found from {Path.cwd()}"

CACHE = ROOT / "cache"
FIGS = ROOT / "figs"
CACHE.mkdir(exist_ok=True)
FIGS.mkdir(exist_ok=True)

print("ROOT :", ROOT)
print("CACHE:", CACHE)
print("FIGS :", FIGS)

## 0.1 - Load the three CSVs

`ossl_tp_metadonnees.csv` is deliberately **not** loaded: its 11 columns are already a strict
subset of `ossl_tp_dataset.csv`, so it would only be a second copy of the same values.

In [ ]:
FILES = {
    "dataset":  "ossl_tp_dataset.csv",   # metadata + 421 spectral columns
    "soillab":  "ossl_tp_soillab.csv",   # 58 laboratory properties
    "soilsite": "ossl_tp_soilsite.csv",  # 36 site / provenance columns
}

raw = {}
for name, fname in FILES.items():
    t0 = time.time()
    raw[name] = pd.read_csv(ROOT / fname, low_memory=False)
    print(f"{name:9s} {str(raw[name].shape):>14s}  "
          f"{(ROOT / fname).stat().st_size / 1e6:7.1f} MB  "
          f"{time.time() - t0:5.1f} s")

dataset, soillab, soilsite = raw["dataset"], raw["soillab"], raw["soilsite"]

## 0.2 - Alignment checks

The three files turn out to be row-for-row aligned on `id`, which makes any join a no-op.
That is a convenience, not a guarantee - so it is asserted rather than assumed. If an
assertion ever fires, switch to an explicit `merge(on="id")`.

In [ ]:
n = len(dataset)

for name, df in raw.items():
    assert len(df) == n, f"{name}: {len(df)} rows, expected {n}"
    assert df["id"].is_unique, f"{name}: duplicate ids"
    assert df["id"].notna().all(), f"{name}: null ids"

aligned = all((raw[name]["id"].values == dataset["id"].values).all() for name in raw)
assert aligned, "row order differs between files -- use merge(on='id') instead of concat"

print(f"{n:,} rows in all three files")
print("ids unique       : yes")
print("identical order  : yes  -> join is a no-op, but use merge(on='id') anyway for safety")

In [ ]:
def merged(*names, on="id"):
    """Join the requested files on `id`. merged('dataset', 'soilsite') -> one DataFrame."""
    out = raw[names[0]]
    for name in names[1:]:
        out = out.merge(raw[name], on=on, suffixes=("", f"_{name}"))
    assert len(out) == n, f"join changed the row count: {len(out)} != {n}"
    return out


# Sanity check only; not kept in memory.
print("merged('dataset', 'soilsite', 'soillab') ->", merged("dataset", "soilsite", "soillab").shape)

## 0.3 - Column groups

Every later section needs the same two objects. Derive them once, from the column names,
so nothing is hard-coded:

- `SPEC` - the 421 spectral column names, in wavelength order
- `WL`   - the matching wavelength axis in nm, for the x-axis of every spectral plot

In [ ]:
SPEC = [c for c in dataset.columns if re.fullmatch(r"L\d+", c)]
SPEC = sorted(SPEC, key=lambda c: int(c[1:]))
WL = np.array([int(c[1:]) for c in SPEC])

META = [c for c in dataset.columns if c not in SPEC]
TARGETS = ["sand", "silt", "clay"]

steps = np.unique(np.diff(WL))
assert steps.tolist() == [5], f"wavelength grid is not regular: {steps}"

print(f"SPEC    : {len(SPEC)} columns, {SPEC[0]} .. {SPEC[-1]}")
print(f"WL      : {WL[0]}-{WL[-1]} nm, constant {steps[0]} nm step")
print(f"META    : {META}")
print(f"TARGETS : {TARGETS}")

## 0.4 - What each file is for

| File | Rows x cols | Role in the lab |
|---|---|---|
| `dataset` | 40 535 x 433 | **The working table.** 12 metadata columns + `L400..L2500`. Sections 1-6 all start here. |
| `soillab` | 40 535 x 58 | Extra laboratory properties (organic carbon, pH, CEC, ...) and the *source* texture columns the targets were derived from. Supporting evidence only. |
| `soilsite` | 40 535 x 36 | Provenance: dataset owner, licence, country, observation date, `layer.upper/lower.depth`, `layer.texture_usda_c`. |

Tasks 1.12-1.14 (map, programme counts, depth) can be answered from either `dataset` or
`soilsite`. **Use `dataset`** and stay consistent, so the numbers in section 1 match the
frame the models are trained on later.

In [ ]:
# The targets in `dataset` are a renormalised copy of these `soillab` columns.
SRC_TEXTURE = ["sand.tot_usda.3a1_wpct", "silt.tot_usda.3a1_wpct", "clay.tot_usda.3a1_wpct"]

print("source texture columns in soillab:")
print(soillab[SRC_TEXTURE].describe().T[["count", "mean", "min", "max"]].to_string())
print()
print("depth in dataset  :", [c for c in META if c.startswith("prof")])
print("depth in soilsite :", [c for c in soilsite.columns if "depth" in c])

## 0.5 - Cache to Parquet

The 139 MB CSV has to be re-parsed by every notebook, in every session: 2 s with the file
warm in the OS cache, tens of seconds cold. Parquet is columnar and typed, so the same
table reloads from 38 MB in ~0.2 s - and a spectra-only read (`columns=SPEC`) is cheaper
still. The timings printed below are measured on this machine.

Values are kept in `float64`: the cache is a faster copy of the data, not a modified one.

In [ ]:
for name, df in raw.items():
    path = CACHE / f"{name}.parquet"
    t0 = time.time()
    df.to_parquet(path, engine="pyarrow", compression="snappy", index=False)
    print(f"{name:9s} -> {path.name:18s} {path.stat().st_size / 1e6:7.1f} MB  "
          f"written in {time.time() - t0:4.1f} s")

In [ ]:
t0 = time.time()
check = pd.read_parquet(CACHE / "dataset.parquet")
t_parquet = time.time() - t0

assert check.shape == dataset.shape
assert list(check.columns) == list(dataset.columns)
assert np.allclose(check[SPEC].to_numpy(), dataset[SPEC].to_numpy(), equal_nan=True)

t0 = time.time()
_ = pd.read_parquet(CACHE / "dataset.parquet", columns=SPEC)
t_spec = time.time() - t0

print("round-trip is exact: yes")
print(f"full reload from parquet   : {t_parquet:4.1f} s")
print(f"spectra-only from parquet  : {t_spec:4.1f} s")
del check

## 0.6 - Figure output

The lab's rule is *one deliverable per task*. `save_fig("1.3")` writes `figs/fig_1_3.png`,
which makes each deliverable findable by its task number.

In [ ]:
plt.rcParams.update({
    "figure.figsize": (9, 5),
    "figure.dpi": 110,
    "savefig.dpi": 150,
    "savefig.bbox": "tight",
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})


def save_fig(task, fig=None):
    """save_fig('1.3') -> figs/fig_1_3.png"""
    fig = fig or plt.gcf()
    path = FIGS / f"fig_{str(task).replace('.', '_')}.png"
    fig.savefig(path)
    print(f"saved {path.relative_to(ROOT)}")
    return path


fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(WL, dataset[SPEC].iloc[0].to_numpy(), lw=1)
ax.set(xlabel="wavelength (nm)", ylabel="reflectance",
       title="save_fig() smoke test - not a deliverable")
save_fig("0.6")
plt.show()

## 0.7 - Two facts to carry into section 5

Both are properties of the data, not of any model, so they are established here - before
any score exists to argue with them.

In [ ]:
print("A. programme vs instrument\n")
print(pd.crosstab(dataset["programme"], dataset["instrument"]).to_string())
print("""
   Each programme uses exactly one instrument and each instrument serves exactly one
   programme: the two are perfectly confounded. Leave-one-programme-out in task 5.1 is
   therefore also leave-one-instrument-out -- and, per the 1.12 map, leave-one-region-out.
   Any drop measured there has three tangled causes and must be reported as such.
""")

s = dataset[TARGETS].sum(axis=1)
print("B. sand + silt + clay\n")
print(f"   min {s.min():.4f}   mean {s.mean():.4f}   max {s.max():.4f}")
print("""
   The targets are already closed: they sum to 100, not to 1. So task 4.3's '2% from 1'
   means 2 points away from 100, and any gap you measure there comes purely from fitting
   the three targets independently -- nothing in the data creates it.
""")

## 0.8 - Bootstrap for the later notebooks

Step 0 is done. Open each later notebook with this cell instead of re-reading the CSVs:

```python
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "cache" / "dataset.parquet").exists():
    ROOT = ROOT.parent
CACHE, FIGS = ROOT / "cache", ROOT / "figs"

dataset = pd.read_parquet(CACHE / "dataset.parquet")
SPEC = sorted((c for c in dataset.columns if re.fullmatch(r"L\d+", c)), key=lambda c: int(c[1:]))
WL = np.array([int(c[1:]) for c in SPEC])
TARGETS = ["sand", "silt", "clay"]

def save_fig(task, fig=None):
    path = FIGS / f"fig_{str(task).replace('.', '_')}.png"
    (fig or plt.gcf()).savefig(path, dpi=150, bbox_inches="tight")
    return path
```

Note for task 1.10: neither `mpltern` nor `python-ternary` is installed. The texture
triangle can be drawn with a plain barycentric transform, so no install is required:

```python
x = 0.5 * (2 * silt + clay) / 100
y = (np.sqrt(3) / 2) * clay / 100
```

**Next:** section 1, tasks 1.1-1.15.